# Use Case - Cross-Service Analysis.ipynb

Compare metering data across two Sovereign Core services — **cluster-as-a-service** and **servicebrokercore** — to identify tenant consumption patterns, usage trends, platform utilisation, and estimated revenue.

| Section | Question answered |
|---|---|
| 2. Configuration | Which services and data directories to load |
| 3. Load Data | Load and join grouped data for both services |
| 4.1 Tenant Consumption Dashboard | Who is consuming the most — and which resources? |
| 4.2 Usage Trend Analysis | How has consumption evolved? Where are the spikes? |
| 4.3 Service Utilisation Breakdown | Which service is driving overall platform load? |
| 4.4 Revenue Estimation | What does each tenant owe under PayGo and Per-Instance models? |

**Prerequisites:** Run `Fetch - Usage Data.ipynb` against both services, or use the included cross-service sample data by setting the paths in `.env` as shown in Section 2.


## 1. Imports

In [1]:
import json
import os
import pathlib

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dotenv import load_dotenv
import ipywidgets as widgets
from IPython.display import display, clear_output


## 2. Configuration

Set `APP_DOMAIN`, `SERVICE_ID_A`, and `SERVICE_ID_B` in `.env` to point at your deployment. The notebook loads grouped data for **both services** from `data/<APP_DOMAIN>/`.

To use the included cross-service sample data, add to `.env`:

```
APP_DOMAIN=apps.cluster.url.com
SERVICE_ID_A=cluster-as-a-service
SERVICE_ID_B=servicebrokercore
```

| Variable | Default | Description |
|---|---|---|
| `APP_DOMAIN` | `apps.cluster.url.com` | Cluster domain |
| `SERVICE_ID_A` | `cluster-as-a-service` | First service |
| `SERVICE_ID_B` | `servicebrokercore` | Second service |


In [2]:
if pathlib.Path(".env").exists():
    load_dotenv(".env", override=False)
else:
    load_dotenv(".env.template", override=False)

# ── Defaults ──────────────────────────────────────────────────────────────────
_DEMO_DOMAIN  = "apps.cluster.url.com"
_DEMO_SVC_A   = "cluster-as-a-service"
_DEMO_SVC_B   = "servicebrokercore"

app_domain    = os.getenv("APP_DOMAIN",    _DEMO_DOMAIN).strip()
service_id_a  = os.getenv("SERVICE_ID_A",  _DEMO_SVC_A).strip()
service_id_b  = os.getenv("SERVICE_ID_B",  _DEMO_SVC_B).strip()

GRP_DIR_A = pathlib.Path(f"data/{app_domain}/{service_id_a}/grouped")
GRP_DIR_B = pathlib.Path(f"data/{app_domain}/{service_id_b}/grouped")

# ── Pricing rates (for Section 4.4) ──────────────────────────────────────────
_style      = {"description_width": "180px"}
_layout     = widgets.Layout(width="480px")
_btn_layout = widgets.Layout(width="220px", height="36px", margin="12px 0 0 184px")

w_paygo_rate    = widgets.FloatText(value=0.50,  description="PayGo ($ / 1M api_calls):", style=_style, layout=_layout)
w_instance_rate = widgets.FloatText(value=200.0, description="Per-instance ($ / period):", style=_style, layout=_layout)
w_btn = widgets.Button(description="Apply", button_style="primary", icon="check", layout=_btn_layout)
w_out = widgets.Output()

PAYGO_RATE    = w_paygo_rate.value    / 1_000_000   # $ per single api_call
INSTANCE_RATE = w_instance_rate.value               # $ per instance-period

def _apply(_):
    global PAYGO_RATE, INSTANCE_RATE
    PAYGO_RATE    = w_paygo_rate.value    / 1_000_000
    INSTANCE_RATE = w_instance_rate.value
    with w_out:
        clear_output(wait=True)
        _print_status()

w_btn.on_click(_apply)

def _print_status():
    for lbl, d in [(service_id_a, GRP_DIR_A), (service_id_b, GRP_DIR_B)]:
        n = len(list(d.glob("*.json"))) if d.exists() else 0
        icon = "✓" if n else "⚠"
        print(f"  {icon} {lbl:<30s}  {d}  ({n} files)")
    print(f"\n  PayGo rate      : ${w_paygo_rate.value:.2f} per 1M api_calls")
    print(f"  Per-instance    : ${w_instance_rate.value:.2f} per instance per period")

display(widgets.VBox([
    widgets.HTML(
        f"<b style='font-size:13px'>Services</b><br>"
        f"<span style='font-family:monospace;font-size:12px;color:#555'>"
        f"A&nbsp;&nbsp;=&nbsp;{service_id_a}<br>"
        f"B&nbsp;&nbsp;=&nbsp;{service_id_b}<br>"
        f"APP_DOMAIN&nbsp;=&nbsp;{app_domain}"
        "</span>"
    ),
    w_out,
    widgets.HTML("<b style='font-size:13px'>Pricing rates (Section 4.4)</b>"),
    w_paygo_rate, w_instance_rate, w_btn,
]))

with w_out:
    _print_status()


## 3. Load Data

Loads **grouped-by-tenant** files for every metric from both service directories. All data is joined into two DataFrames — one per service — keyed on `(groupTenantId, periodStart)`.

| Variable | Source | Contents |
|---|---|---|
| `grp_a` | `GRP_DIR_A` | All grouped-by-tenant metrics for service A |
| `grp_b` | `GRP_DIR_B` | All grouped-by-tenant metrics for service B |


In [3]:
def _load_all_grouped_by_tenant(directory: pathlib.Path, service_id: str) -> pd.DataFrame:
    """Load all groupByTenant JSON files into one wide DataFrame indexed by
    (groupTenantId, date).  Each metric becomes its own column."""
    frames = []
    for _p in sorted(directory.glob("*.json")):
        try:
            _d = json.loads(_p.read_text())
            if _d.get("params", {}).get("groupBy") != "groupByTenant":
                continue
            _metric    = _d["params"]["metricId"]
            _transform = _d["params"]["transform"]
            _periods   = _d.get("aggregatedMeteredUsagePeriods", [])
            if not _periods:
                continue
            df = pd.DataFrame(_periods)
            df["date"]      = pd.to_datetime(df["periodStart"], unit="ms", utc=True).dt.date
            df["metricId"]  = _metric
            df["transform"] = _transform
            df["service"]   = service_id
            df = df.rename(columns={"periodQuantity": "value", "groupTenantId": "tenant"})
            frames.append(df[["date", "tenant", "metricId", "transform", "service", "value"]])
        except Exception:
            pass
    if not frames:
        return pd.DataFrame(columns=["date", "tenant", "metricId", "transform", "service", "value"])
    return pd.concat(frames, ignore_index=True)


def _load_all_grouped_by_instance(directory: pathlib.Path, service_id: str) -> pd.DataFrame:
    """Load all groupByInstance JSON files — needed for per-instance billing counts."""
    frames = []
    for _p in sorted(directory.glob("*.json")):
        try:
            _d = json.loads(_p.read_text())
            if _d.get("params", {}).get("groupBy") != "groupByInstance":
                continue
            _metric    = _d["params"]["metricId"]
            _transform = _d["params"]["transform"]
            _periods   = _d.get("aggregatedMeteredUsagePeriods", [])
            if not _periods:
                continue
            df = pd.DataFrame(_periods)
            df["date"]       = pd.to_datetime(df["periodStart"], unit="ms", utc=True).dt.date
            df["metricId"]   = _metric
            df["transform"]  = _transform
            df["service"]    = service_id
            df = df.rename(columns={
                "periodQuantity":  "value",
                "groupTenantId":   "tenant",
                "groupInstanceId": "instance",
            })
            frames.append(df[["date", "tenant", "instance", "metricId", "transform", "service", "value"]])
        except Exception:
            pass
    if not frames:
        return pd.DataFrame(columns=["date", "tenant", "instance", "metricId", "transform", "service", "value"])
    return pd.concat(frames, ignore_index=True)


# ── Load all data ─────────────────────────────────────────────────────────────
grp_a      = _load_all_grouped_by_tenant(GRP_DIR_A, service_id_a)
grp_b      = _load_all_grouped_by_tenant(GRP_DIR_B, service_id_b)
grp_a_inst = _load_all_grouped_by_instance(GRP_DIR_A, service_id_a)
grp_b_inst = _load_all_grouped_by_instance(GRP_DIR_B, service_id_b)

# Combined long-form DataFrame (both services)
grp_all      = pd.concat([grp_a,      grp_b],      ignore_index=True)
grp_all_inst = pd.concat([grp_a_inst, grp_b_inst], ignore_index=True)

# ── Colour palette — one colour per tenant, consistent across all charts ──────
_tenants = sorted(grp_all["tenant"].unique())
_palette = px.colors.qualitative.Bold
TENANT_COLOURS = {t: _palette[i % len(_palette)] for i, t in enumerate(_tenants)}

# ── Short label helper ────────────────────────────────────────────────────────
def _short(tid: str) -> str:
    return tid if len(tid) <= 18 else tid[:16] + "…"

# ── Status ────────────────────────────────────────────────────────────────────
for svc, df in [(service_id_a, grp_a), (service_id_b, grp_b)]:
    metrics  = sorted(df["metricId"].unique())
    tenants  = sorted(df["tenant"].unique())
    n_days   = df["date"].nunique()
    print(f"✓ {svc}")
    print(f"  {len(df):,} rows  |  {len(tenants)} tenants  |  {len(metrics)} metrics  |  {n_days} days")
    print(f"  metrics : {', '.join(metrics)}")
    print(f"  tenants : {', '.join(_short(t) for t in tenants)}")
    print()


✓ cluster-as-a-service
  750 rows  |  5 tenants  |  5 metrics  |  30 days
  metrics : api_calls, cpu_cores, gpu_count, instances, users
  tenants : BANK_CORP, HEALTHNET_ORG, MANUFACT_WORKS, MOTORVAULT_INC, RETAILPLEX_CO

✓ servicebrokercore
  750 rows  |  5 tenants  |  5 metrics  |  30 days
  metrics : api_calls, cpu_cores, gpu_count, instances, users
  tenants : BANK_CORP, HEALTHNET_ORG, MANUFACT_WORKS, MOTORVAULT_INC, RETAILPLEX_CO



## 4. Charts

### 4.1 Tenant Consumption Dashboard

*Which tenants consume the most resources — and does the pattern differ between services?*

### Computation

Loads `groupByTenant` files for `cpu_cores`, `gpu_count`, and `instances` from both service directories.
For each metric, computes the 30-day mean of `periodQuantity` per tenant per service.
Bars are rendered in pairs (one per service) for each tenant, sorted by combined descending total,
so the highest-consuming tenants appear first.

In [4]:
fig_41 = None

# ── Metric selector ───────────────────────────────────────────────────────────
_41_metrics   = ["cpu_cores", "gpu_count", "instances"]
_41_available = [m for m in _41_metrics if m in grp_all["metricId"].values]
w_41_metric   = widgets.ToggleButtons(
    options=_41_available,
    value=_41_available[0] if _41_available else None,
    description="Metric:",
    button_style="",
    style={"description_width": "60px", "button_width": "120px"},
)
w_41_out = widgets.Output()

def _draw_41(metric):
    global fig_41
    with w_41_out:
        clear_output(wait=True)
        if not metric:
            print("⚠ No resource metrics available.")
            return

        # ── Per-tenant 30-day mean for each service ────────────────────────
        sub = grp_all[grp_all["metricId"] == metric].copy()
        tenant_svc = (
            sub.groupby(["tenant", "service"])["value"]
            .mean()
            .reset_index()
            .rename(columns={"value": "avg_value"})
        )
        tenant_svc["label"] = tenant_svc["tenant"].apply(_short)
        tenant_svc = tenant_svc.sort_values("avg_value", ascending=False)

        transform = sub["transform"].iloc[0] if not sub.empty else "avg"

        fig_41 = go.Figure()
        bar_width = 0.35

        # One bar cluster per tenant; two bars (services A and B) side by side
        unique_tenants = sorted(tenant_svc["tenant"].unique(),
                                key=lambda t: -tenant_svc.loc[tenant_svc["tenant"]==t, "avg_value"].sum())
        x_pos = list(range(len(unique_tenants)))
        offset_a, offset_b = -bar_width / 2, bar_width / 2

        for svc, offset, pattern in [
            (service_id_a, offset_a, ""),
            (service_id_b, offset_b, "/"),
        ]:
            vals  = []
            texts = []
            for t in unique_tenants:
                row = tenant_svc[(tenant_svc["tenant"] == t) & (tenant_svc["service"] == svc)]
                v   = row["avg_value"].values[0] if not row.empty else 0
                vals.append(v)
                texts.append(f"{v:,.1f}")
            colours = [TENANT_COLOURS.get(t, "#888") for t in unique_tenants]
            fig_41.add_trace(go.Bar(
                x=[p + offset for p in x_pos],
                y=vals,
                name=svc,
                marker_color=colours,
                marker_pattern_shape=["", pattern][int(pattern == "/")],
                text=texts,
                textposition="outside",
                textfont_size=10,
                width=bar_width,
                legendgroup=svc,
                showlegend=True,
                customdata=[[t, svc] for t in unique_tenants],
                hovertemplate=(
                    "<b>%{customdata[0]}</b><br>"
                    "Service: %{customdata[1]}<br>"
                    f"{metric}: %{{y:,.1f}}<extra></extra>"
                ),
            ))

        fig_41.update_layout(
            title=dict(
                text=f"<b>Tenant Consumption — {metric} ({transform})</b><br>"
                     "<sup>30-day average per tenant · solid = cluster-as-a-service · hatched = servicebrokercore</sup>",
                font_size=14,
            ),
            xaxis=dict(
                tickvals=x_pos,
                ticktext=[_short(t) for t in unique_tenants],
                tickangle=-20,
            ),
            yaxis_title=f"Avg {metric}",
            barmode="overlay",
            bargap=0.25,
            height=480,
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
            margin=dict(t=100, b=60),
            plot_bgcolor="#fafafa",
        )
        fig_41.show();

# widgets.interactive_output(_draw_41, {"metric": w_41_metric})
# display(widgets.VBox([w_41_metric, w_41_out]))
# _draw_41(w_41_metric.value);


### Chart Guide

**Purpose:** Compares infrastructure resource consumption across tenants for both services side-by-side.
Operators can immediately spot high-consumption environments and see whether a tenant's load profile is symmetric (heavy on both services) or skewed (e.g. cluster-heavy but broker-light).

**Chart Attributes**

| | |
|---|---|
| **Chart type** | Grouped bar chart |
| **X axis** | Tenant (sorted by total consumption, descending) |
| **Y axis** | 30-day average of the selected metric |
| **Bars** | Solid = `cluster-as-a-service` · Hatched = `servicebrokercore` |
| **Colours** | One colour per tenant, consistent across all charts |
| **Selector** | Toggle button — switches metric between `cpu_cores`, `gpu_count`, `instances` |
| **Hover** | Tenant ID · Service · exact metric value |

**Insights:** A tall solid bar with a short hatched bar indicates heavy infrastructure use with lighter brokered service consumption — typical of IoT or GPU-intensive workloads (`MANUFACT_WORKS`).
Symmetric bars (e.g. `BANK_CORP`, `RETAILPLEX_CO`) mean the tenant drives load equally across both platform layers.
`BANK_CORP` shows `instances = 2` on `cluster-as-a-service` throughout — a compliance-mandated fixed allocation.

In [5]:
if fig_41 is not None:
    fig_41.show()
widgets.interactive_output(_draw_41, {"metric": w_41_metric})
display(widgets.VBox([w_41_metric, w_41_out]))
_draw_41(w_41_metric.value);


### 4.2 Usage Trend Analysis

*How has platform consumption evolved over the 30-day window? Where are the spikes?*

### Computation

Loads `groupByTenant` files for `api_calls` (transform: `sum`) and `users` (transform: `avg`).
Daily `periodQuantity` values are plotted per tenant as a time-series.
Both service panels share the same Y axis so relative magnitudes are directly comparable.

In [6]:
fig_42 = None

# ── Metric selector (api_calls or users) ──────────────────────────────────────
_42_opts = [m for m in ["api_calls", "users"] if m in grp_all["metricId"].values]
w_42_metric = widgets.ToggleButtons(
    options=_42_opts,
    value=_42_opts[0] if _42_opts else None,
    description="Metric:",
    button_style="",
    style={"description_width": "60px", "button_width": "120px"},
)
w_42_out = widgets.Output()

def _draw_42(metric):
    global fig_42
    with w_42_out:
        clear_output(wait=True)
        if not metric:
            print("⚠ No trend metrics available.")
            return

        sub = grp_all[grp_all["metricId"] == metric].copy()
        sub["date"] = pd.to_datetime(sub["date"])
        transform   = sub["transform"].iloc[0] if not sub.empty else ""
        services    = sorted(sub["service"].unique())

        fig_42 = make_subplots(
            rows=1, cols=len(services),
            shared_yaxes=True,
            subplot_titles=[f"<b>{s}</b>" for s in services],
            horizontal_spacing=0.06,
        )

        for col_idx, svc in enumerate(services, start=1):
            svc_df = sub[sub["service"] == svc]
            tenants = sorted(svc_df["tenant"].unique())
            for tenant in tenants:
                t_df = svc_df[svc_df["tenant"] == tenant].sort_values("date")
                fig_42.add_trace(go.Scatter(
                    x=t_df["date"],
                    y=t_df["value"],
                    mode="lines+markers",
                    name=_short(tenant),
                    line=dict(color=TENANT_COLOURS.get(tenant, "#888"), width=2),
                    marker=dict(size=5),
                    legendgroup=tenant,
                    showlegend=(col_idx == 1),
                    customdata=[[tenant, svc]] * len(t_df),
                    hovertemplate=(
                        "<b>%{customdata[0]}</b><br>"
                        "Service: %{customdata[1]}<br>"
                        "Date: %{x|%Y-%m-%d}<br>"
                        f"{metric}: %{{y:,.0f}}<extra></extra>"
                    ),
                ), row=1, col=col_idx)

        fig_42.update_layout(
            title=dict(
                text=(
                    f"<b>Usage Trend — {metric} ({transform})</b><br>"
                    "<sup>Daily values per tenant · both services · hover for exact values</sup>"
                ),
                font_size=14,
            ),
            height=420,
            legend=dict(
                title="Tenant",
                orientation="v",
                x=1.01, y=1,
                xanchor="left",
            ),
            hovermode="x unified",
            plot_bgcolor="#fafafa",
            margin=dict(t=100, r=160),
        )
        fig_42.update_yaxes(title_text=f"{metric} ({transform})", col=1)
        fig_42.show();




### Chart Guide

**Purpose:** Reveals growth patterns, recurring peaks, and anomalous one-day spikes across tenants and services.
A sudden spike on the same day across both service panels confirms a real platform event rather than a data artefact.

**Chart Attributes**

| | |
|---|---|
| **Chart type** | Multi-line time-series (one line per tenant) |
| **Panels** | Left = `cluster-as-a-service` · Right = `servicebrokercore` (shared Y axis) |
| **X axis** | Date (daily resolution) |
| **Y axis** | Metric value — `sum` for `api_calls`, `avg` for `users` |
| **Colours** | One colour per tenant, consistent across all charts |
| **Selector** | Toggle button — switches between `api_calls` and `users` |
| **Hover** | Tenant · Service · Date · exact value |

**Insights:** Sharp single-day spikes correspond to encoded cross-service events:
- **Day 8** — `BANK_CORP` quarter-end: `servicebrokercore` api_calls spike; `cluster-as-a-service` cpu_cores spike simultaneously.
- **Day 20** — `RETAILPLEX_CO` flash sale: largest api_calls spike across both services.
- **Day 24** — `MANUFACT_WORKS` IoT telemetry storm: api_calls spike on broker; gpu_count spike on cluster.

Gradually rising lines (`MOTORVAULT_INC`) indicate organic growth — not incidents.

In [7]:
if fig_42 is not None:
    fig_42.show()
widgets.interactive_output(_draw_42, {"metric": w_42_metric})
display(widgets.VBox([w_42_metric, w_42_out]))
_draw_42(w_42_metric.value);

### 4.3 Service Utilisation Breakdown

*Which platform service is driving the most load — and does this vary by tenant?*

### Computation

Aggregates total `api_calls (sum)` per tenant per service over the full 30-day window.
Two views are produced from the same data: absolute totals (left panel) and percentage share per tenant (right panel).
Tenants are ordered by combined total api_calls descending.

In [12]:
fig_43 = None

_43_metric = "api_calls"
sub_43 = grp_all[grp_all["metricId"] == _43_metric].copy()

# Total per tenant per service
tenant_svc_43 = (
    sub_43.groupby(["tenant", "service"])["value"]
    .sum()
    .reset_index()
    .rename(columns={"value": "total"})
)

# Compute share within each tenant
tenant_totals_43 = tenant_svc_43.groupby("tenant")["total"].transform("sum")
tenant_svc_43["share_pct"] = tenant_svc_43["total"] / tenant_totals_43 * 100

tenant_order = (
    tenant_svc_43.groupby("tenant")["total"].sum()
    .sort_values(ascending=False)
    .index.tolist()
)

_svc_colours = {
    service_id_a: "#2563EB",   # blue — cluster
    service_id_b: "#7C3AED",   # purple — broker
}

fig_43 = make_subplots(
    rows=1, cols=2,
    subplot_titles=["<b>Absolute api_calls (30-day total)</b>", "<b>Service share per tenant (%)</b>"],
    horizontal_spacing=0.10,
)

for svc in [service_id_a, service_id_b]:
    svc_df = tenant_svc_43[tenant_svc_43["service"] == svc].set_index("tenant")
    absolutes = [svc_df.loc[t, "total"]     if t in svc_df.index else 0 for t in tenant_order]
    shares    = [svc_df.loc[t, "share_pct"] if t in svc_df.index else 0 for t in tenant_order]
    short_labels = [_short(t) for t in tenant_order]

    fig_43.add_trace(go.Bar(
        x=short_labels,
        y=absolutes,
        name=svc,
        marker_color=_svc_colours.get(svc, "#888"),
        legendgroup=svc,
        showlegend=True,
        hovertemplate=f"<b>%{{x}}</b><br>{svc}<br>api_calls: %{{y:,.0f}}<extra></extra>",
    ), row=1, col=1)

    fig_43.add_trace(go.Bar(
        x=short_labels,
        y=shares,
        name=svc,
        marker_color=_svc_colours.get(svc, "#888"),
        legendgroup=svc,
        showlegend=False,
        hovertemplate=f"<b>%{{x}}</b><br>{svc}<br>Share: %{{y:.1f}}%<extra></extra>",
    ), row=1, col=2)

fig_43.update_layout(
    barmode="stack",
    title=dict(
        text="<b>Service Utilisation Breakdown — api_calls</b><br>"
             "<sup>Tenants ranked by total 30-day api_calls · blue = cluster-as-a-service · purple = servicebrokercore</sup>",
        font_size=14,
    ),
    height=440,
    legend=dict(orientation="h", yanchor="bottom", y=1.04, xanchor="right", x=1),
    plot_bgcolor="#fafafa",
    margin=dict(t=110),
)
fig_43.update_yaxes(title_text="api_calls (sum)", col=1)
fig_43.update_yaxes(title_text="Share (%)", range=[0, 100], col=2)
fig_43.show();


### Chart Guide

**Purpose:** Shows both the raw volume each tenant generates and how that volume is split between the two platform services.
Helps the MSP understand which service is doing more work and whether that distribution is uniform or tenant-specific.

**Chart Attributes**

| | |
|---|---|
| **Chart type** | Stacked bar chart (two panels) |
| **Left panel** | Absolute `api_calls` total per tenant · stacked by service |
| **Right panel** | 100% share per tenant · shows proportional split |
| **X axis** | Tenant (sorted by total api_calls, descending) |
| **Y axis (left)** | api_calls (sum over 30 days) |
| **Y axis (right)** | Percentage share (0–100%) |
| **Colours** | Blue = `cluster-as-a-service` · Purple = `servicebrokercore` |
| **Hover** | Tenant · Service · value / share % |

**Insights:** The left panel shows volume dominance — `RETAILPLEX_CO` and `MANUFACT_WORKS` are typically the highest-throughput tenants.
The right panel exposes structural patterns: `MANUFACT_WORKS` skews heavily blue because its IoT edge workloads are cluster-managed, not brokered.
Tenants with a near-50/50 split (e.g. `BANK_CORP`) consume both platform layers equally.

In [9]:
if fig_43 is not None:
    fig_43.show()


### 4.4 Revenue Estimation

*What would each tenant owe under different billing models?*

### Computation

**PayGo** — sums `api_calls` per tenant per service over 30 days, then multiplies by `PAYGO_RATE` ($ per call).
**Per-Instance** — counts distinct `instanceId` values per tenant per service from `groupByInstance` files,
then multiplies by `INSTANCE_RATE` ($ per instance per period).
Rates are read from the live widget values in Section 2 — adjust and re-run this cell to see the impact immediately.

In [10]:
fig_44 = None

# ── PayGo: api_calls (sum) × PAYGO_RATE ──────────────────────────────────────
paygo_df = (
    grp_all[grp_all["metricId"] == "api_calls"]
    .groupby(["tenant", "service"])["value"]
    .sum()
    .reset_index()
    .rename(columns={"value": "total_api_calls"})
)
paygo_df["paygo_revenue"] = paygo_df["total_api_calls"] * PAYGO_RATE

# ── Per-Instance: distinct instances × INSTANCE_RATE ─────────────────────────
inst_count_df = (
    grp_all_inst[grp_all_inst["metricId"] == "instances"]
    .groupby(["tenant", "service"])["instance"]
    .nunique()
    .reset_index()
    .rename(columns={"instance": "instance_count"})
)
inst_count_df["instance_revenue"] = inst_count_df["instance_count"] * INSTANCE_RATE

# ── Merge models ──────────────────────────────────────────────────────────────
rev_df = pd.merge(paygo_df[["tenant", "service", "paygo_revenue"]],
                  inst_count_df[["tenant", "service", "instance_revenue"]],
                  on=["tenant", "service"], how="outer").fillna(0)

tenant_order_rev = (
    rev_df.groupby("tenant")[["paygo_revenue", "instance_revenue"]].sum()
    .sum(axis=1)
    .sort_values(ascending=False)
    .index.tolist()
)

fig_44 = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        f"<b>Model 1 — PayGo</b><br>"
        f"<sup>${w_paygo_rate.value:.2f} per 1M api_calls</sup>",
        f"<b>Model 2 — Per-Instance Flat Rate</b><br>"
        f"<sup>${w_instance_rate.value:.2f} per instance per period</sup>",
    ],
    horizontal_spacing=0.12,
)

for svc in [service_id_a, service_id_b]:
    svc_rev = rev_df[rev_df["service"] == svc].set_index("tenant")
    col_colour = _svc_colours.get(svc, "#888")
    short_labels = [_short(t) for t in tenant_order_rev]

    paygo_vals = [svc_rev.loc[t, "paygo_revenue"]    if t in svc_rev.index else 0 for t in tenant_order_rev]
    inst_vals  = [svc_rev.loc[t, "instance_revenue"] if t in svc_rev.index else 0 for t in tenant_order_rev]

    fig_44.add_trace(go.Bar(
        x=short_labels, y=paygo_vals,
        name=svc, marker_color=col_colour,
        legendgroup=svc, showlegend=True,
        text=[f"${v:,.0f}" for v in paygo_vals],
        textposition="outside", textfont_size=9,
        hovertemplate=f"<b>%{{x}}</b><br>{svc}<br>PayGo: $%{{y:,.2f}}<extra></extra>",
    ), row=1, col=1)

    fig_44.add_trace(go.Bar(
        x=short_labels, y=inst_vals,
        name=svc, marker_color=col_colour,
        legendgroup=svc, showlegend=False,
        text=[f"${v:,.0f}" for v in inst_vals],
        textposition="outside", textfont_size=9,
        hovertemplate=f"<b>%{{x}}</b><br>{svc}<br>Per-Instance: $%{{y:,.2f}}<extra></extra>",
    ), row=1, col=2)

fig_44.update_layout(
    barmode="group",
    title=dict(
        text="<b>Revenue Estimation by Tenant and Billing Model</b><br>"
             "<sup>Adjust rates in Section 2 and re-run · blue = cluster-as-a-service · purple = servicebrokercore</sup>",
        font_size=14,
    ),
    height=480,
    legend=dict(orientation="h", yanchor="bottom", y=1.08, xanchor="right", x=1),
    plot_bgcolor="#fafafa",
    margin=dict(t=120, b=60),
)
fig_44.update_yaxes(title_text="Estimated Revenue ($)", col=1)
fig_44.update_yaxes(title_text="Estimated Revenue ($)", col=2)
fig_44.show();


### Chart Guide

**Purpose:** Translates raw consumption into estimated revenue under two pricing models, for each service independently.
Helps the MSP compare how different billing structures affect the cost distribution across tenants.

**Chart Attributes**

| | |
|---|---|
| **Chart type** | Grouped bar chart (two panels) |
| **Left panel** | Model 1 — PayGo · `api_calls × rate per 1M calls` |
| **Right panel** | Model 2 — Per-Instance Flat Rate · `instance count × rate per period` |
| **X axis** | Tenant (sorted by combined revenue, descending) |
| **Y axis** | Estimated revenue in USD |
| **Bars** | Blue = `cluster-as-a-service` · Purple = `servicebrokercore` |
| **Bar label** | Dollar amount (e.g. `$1,240`) |
| **Hover** | Tenant · Service · estimated revenue |

**Insights:** PayGo (left) scales linearly with usage — bar heights mirror the utilisation breakdown in 4.3.
Per-Instance (right) charges for provisioned capacity regardless of actual usage:
`RETAILPLEX_CO` pays more here because it maintains four instances even on quiet days,
while `HEALTHNET_ORG` pays less despite high GPU usage (few instances, bursty workload).
Adjust the rates in Section 2 and re-run to explore the pricing sensitivity.

In [11]:
if fig_44 is not None:
    fig_44.show()
